In [ ]:
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.transform import array_bounds
import matplotlib.pyplot as plt
import osmnx as ox



In [ ]:


tif_path = "Yixing_data\chn_level0_100m_2000_2020.tif"  # 换成你的 WorldPop tif

# 1) 直接用地名取行政边界（建议用“结构化查询”减少歧义）
place = {"city": "Yixing", "state": "Jiangsu", "country": "China"}
yixing = ox.geocode_to_gdf(place)  # 返回 GeoDataFrame（通常是面边界）

# 2) 用边界裁剪栅格
with rasterio.open(tif_path) as src:
    yixing = yixing.to_crs(src.crs)
    out, out_transform = mask(src, yixing.geometry, crop=True)
    data = out[0].astype("float32")
    if src.nodata is not None:
        data[data == src.nodata] = np.nan
    h, w = data.shape
    left, bottom, right, top = array_bounds(h, w, out_transform)

# 3) 画图（log1p 让城镇/乡村差异更明显）
img = np.log1p(data)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(img, extent=(left, right, bottom, top), origin="upper")
yixing.boundary.plot(ax=ax, linewidth=1)
ax.set_title("宜兴市人口分布（WorldPop，log1p）")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="log(1 + people per pixel)")
plt.tight_layout()
plt.show()